<a href="https://colab.research.google.com/github/Arfa-Tariq/AstroPlanner-AI/blob/main/notebooks/05_field_of_view_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AstroPlanner AI: Field-of-View Analyzer
## Phase 5 — Computes imaging field of view and determines whether targets fit perfectly, are too large, or are too small

--------------------------------------------------------------------------
### Objective
Combine the user's telescope focal length with their camera's sensor size and pixel size to compute:
1. Imaging field of view (arcminutes), and
2. Pixel scale / image sampling (arcsec per pixel)

...then use those to classify how well each recommended target (from notebook 04) fits the frame. Solar system bodies (planets, Moon) are handled separately, since they're imaged with a fundamentally different technique (high-frame-rate video + stacking) rather than single wide-field frames.

If the user never provided camera specs in notebook 01, this notebook degrades gracefully: it skips FoV math entirely and tags every object `fov_fit: "no_camera"` so downstream consumers (Phase 6 chatbot) can explain why, rather than crashing or guessing.

## Setup

In [9]:
!pip install requests -q

import sys, os, json
from collections import Counter
from datetime import date as date_type
from google.colab import drive

drive.mount('/content/drive')

!git clone https://github.com/Arfa-Tariq/AstroPlanner-AI.git 2>/dev/null || git -C /content/AstroPlanner-AI pull

sys.path.append('/content/AstroPlanner-AI/src')

DATA_DIR = '/content/drive/MyDrive/AstroPlanner/data'
os.makedirs(DATA_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.


In [10]:
from models import WeeklyPlanRequest, UserProfile, CameraSpec, TelescopeSpec

## Load upstream outputs (notebooks 01, 04)

In [11]:
with open(f'{DATA_DIR}/current_request.json') as f:
    plan_request = WeeklyPlanRequest.model_validate_json(f.read())

with open(f'{DATA_DIR}/weekly_recommendations.json') as f:
    weekly_recommendations = json.load(f)

user = plan_request.user
has_camera = user.camera is not None

print(f"User          : {user.name}")
print(f"Telescope     : {user.telescope.aperture_mm}mm aperture, {user.telescope.focal_length_mm}mm focal length")
if has_camera:
    print(f"Camera        : {user.camera.sensor_width_mm}x{user.camera.sensor_height_mm}mm sensor, {user.camera.pixel_size_um}\u00b5m pixels")
else:
    print("Camera        : none provided \u2014 FoV analysis will be skipped, visual-only notes will be attached instead")
print(f"Nights loaded : {len(weekly_recommendations)}")

User          : Andrew
Telescope     : 20.0mm aperture, 100.0mm focal length
Camera        : none provided — FoV analysis will be skipped, visual-only notes will be attached instead
Nights loaded : 7


## Field of view + pixel scale calculations

Both use the standard small-angle approximation (`206265"` / `3437.75'` per radian). This holds fine at amateur focal lengths and sensor sizes — it only breaks down at fields of view spanning many tens of degrees, far outside anything a telescope+camera combo here would produce.

In [12]:
ARCMIN_PER_RADIAN = 3437.7468
ARCSEC_PER_RADIAN = 206264.8


def compute_fov_arcmin(sensor_width_mm: float, sensor_height_mm: float, focal_length_mm: float) -> tuple[float, float]:
    """
    FoV (arcmin) = 3437.75 * sensor_dimension_mm / focal_length_mm, applied
    to both sensor axes. Returns (fov_width_arcmin, fov_height_arcmin).
    """
    fov_w = ARCMIN_PER_RADIAN * sensor_width_mm / focal_length_mm
    fov_h = ARCMIN_PER_RADIAN * sensor_height_mm / focal_length_mm
    return fov_w, fov_h


def compute_pixel_scale_arcsec_per_px(pixel_size_um: float, focal_length_mm: float) -> float:
    """
    Image scale in arcsec/pixel = 206264.8 * (pixel_size_um / 1000) / focal_length_mm.
    Governs whether the setup is oversampled (wasting resolution the
    atmosphere can't deliver anyway) or undersampled (losing detail the
    optics could otherwise resolve).
    """
    pixel_size_mm = pixel_size_um / 1000
    return ARCSEC_PER_RADIAN * pixel_size_mm / focal_length_mm


def classify_sampling(arcsec_per_px: float) -> str:
    """
    Rule-of-thumb bands against typical amateur seeing (~2-3\"):
    - <1.0\"/px    : oversampled  — finer than the atmosphere can resolve
    - 1.0-2.5\"/px : well_matched
    - >2.5\"/px    : undersampled — losing resolution the optics could deliver
    """
    if arcsec_per_px < 1.0:
        return "oversampled"
    if arcsec_per_px <= 2.5:
        return "well_matched"
    return "undersampled"

## Target fit classification

Deep-sky objects use their cataloged `size_arcmin` (from the OpenNGC catalog, notebook 03) against the FoV's *shorter* dimension — that's the binding constraint for framing. Solar system bodies don't carry a `size_arcmin` field at all (see notebook 03's planet output), so they're handled separately with a small table of typical apparent sizes and judged by pixel-scale adequacy instead of frame fit, since planetary/lunar imaging is high-frame-rate video + stacking rather than single-frame framing.

In [13]:
# Rough mean apparent angular size (arcsec) for solar system bodies —
# these vary night to night with orbital distance, but a fixed typical
# value is good enough to steer users toward the right imaging technique
# and isn't recomputed per-night elsewhere in the pipeline.
SOLAR_SYSTEM_TYPICAL_SIZE_ARCSEC = {
    'Moon': 1860.0,     # ~31 arcmin
    'Jupiter': 40.0,
    'Saturn': 17.0,
    'Mars': 10.0,
    'Venus': 25.0,
    'Mercury': 7.0,
}


def classify_deep_sky_fit(object_size_arcmin: "float | None", fov_w_arcmin: float, fov_h_arcmin: float) -> dict:
    """
    Compares a deep-sky object's angular size against the imaging FoV's
    shorter dimension. Returns a verdict plus the underlying ratio, so
    the number is always visible alongside the label rather than hidden
    behind a threshold check.

    Thresholds are about framing quality, not raw geometric fit:
    - ratio > 1.1   : too_large — won't fit in a single frame with margin
    - ratio < 0.10  : too_small — object is a tiny feature in the frame
    - otherwise     : fits_well
    """
    frame_min_dim = min(fov_w_arcmin, fov_h_arcmin)

    if object_size_arcmin is None:
        return {
            'fov_fit': 'unknown',
            'fit_ratio': None,
            'note': 'No cataloged size for this object — fit could not be assessed.',
        }

    ratio = object_size_arcmin / frame_min_dim

    if ratio > 1.1:
        verdict = 'too_large'
        note = "Won't fit in a single frame — consider a mosaic or a shorter focal length."
    elif ratio < 0.10:
        verdict = 'too_small'
        note = 'Appears as a small feature in the frame — consider a longer focal length or a Barlow/reducer as appropriate.'
    else:
        verdict = 'fits_well'
        note = 'Well-framed for this setup.'

    return {
        'fov_fit': verdict,
        'fit_ratio': round(ratio, 3),
        'note': note,
    }


def classify_solar_system_target(obj_name: str, arcsec_per_px: float) -> dict:
    """
    Solar system bodies are always tiny relative to a deep-sky FoV, so
    'does it fit in the frame' isn't the useful question — 'can the pixel
    scale resolve it' is, since planetary/lunar imaging uses high-frame-rate
    capture and stacking rather than single wide-field frames.
    """
    typical_size_arcsec = SOLAR_SYSTEM_TYPICAL_SIZE_ARCSEC.get(obj_name)
    sampling = classify_sampling(arcsec_per_px)

    return {
        'fov_fit': 'planetary_target',
        'fit_ratio': None,
        'typical_angular_size_arcsec': typical_size_arcsec,
        'sampling': sampling,
        'note': (
            'Solar system target — use high-frame-rate planetary/lunar capture '
            'and stacking rather than single-frame deep-sky FoV framing. '
            f"Current pixel scale is {sampling.replace('_', ' ')} for this technique."
        ),
    }

## Apply FoV analysis across the week

In [14]:
def get_weekly_fov_analysis(weekly_recommendations: list, user: UserProfile) -> tuple[list, "dict | None"]:
    """
    Attaches FoV/sampling analysis to every recommended object in every
    night, non-destructively — all upstream fields from notebook 04 are
    preserved via dict unpacking. Returns (weekly, setup_summary);
    setup_summary is None if the user has no camera on file, and in that
    case every object is tagged fov_fit='no_camera' instead of being
    silently dropped, so Phase 6's chatbot can explain why FoV data is
    missing rather than just omitting it.
    """
    if user.camera is None:
        weekly = []
        for night in weekly_recommendations:
            objects = [
                {**obj, 'fov_analysis': {
                    'fov_fit': 'no_camera',
                    'note': 'No camera on file for this user — add camera specs in notebook 01 to enable FoV analysis.',
                }}
                for obj in night['recommended_objects']
            ]
            weekly.append({**night, 'recommended_objects': objects})
        return weekly, None

    fov_w, fov_h = compute_fov_arcmin(
        user.camera.sensor_width_mm, user.camera.sensor_height_mm, user.telescope.focal_length_mm
    )
    arcsec_per_px = compute_pixel_scale_arcsec_per_px(
        user.camera.pixel_size_um, user.telescope.focal_length_mm
    )

    setup_summary = {
        'fov_width_arcmin': round(fov_w, 1),
        'fov_height_arcmin': round(fov_h, 1),
        'fov_diagonal_arcmin': round((fov_w ** 2 + fov_h ** 2) ** 0.5, 1),
        'pixel_scale_arcsec_per_px': round(arcsec_per_px, 2),
        'sampling': classify_sampling(arcsec_per_px),
    }

    weekly = []
    for night in weekly_recommendations:
        objects = []
        for obj in night['recommended_objects']:
            if obj.get('is_solar_system'):
                fov_analysis = classify_solar_system_target(obj['name'], arcsec_per_px)
            else:
                fov_analysis = classify_deep_sky_fit(obj.get('size_arcmin'), fov_w, fov_h)
            objects.append({**obj, 'fov_analysis': fov_analysis})
        weekly.append({**night, 'recommended_objects': objects})

    return weekly, setup_summary

## Run and save

In [15]:
weekly_fov_analysis, setup_summary = get_weekly_fov_analysis(weekly_recommendations, user)

output = {
    'setup_summary': setup_summary,
    'nights': weekly_fov_analysis,
}

with open(f'{DATA_DIR}/weekly_fov_analysis.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)

print(f"Saved to {DATA_DIR}/weekly_fov_analysis.json\n")

if setup_summary:
    print("=== Imaging Setup Summary ===")
    print(f"FoV            : {setup_summary['fov_width_arcmin']}' x {setup_summary['fov_height_arcmin']}' "
          f"(diagonal {setup_summary['fov_diagonal_arcmin']}')")
    print(f"Pixel scale    : {setup_summary['pixel_scale_arcsec_per_px']}\"/px  [{setup_summary['sampling']}]")
else:
    print("No camera on file — FoV analysis skipped, visual-only notes attached to every object.")

Saved to /content/drive/MyDrive/AstroPlanner/data/weekly_fov_analysis.json

No camera on file — FoV analysis skipped, visual-only notes attached to every object.


## Spot check

In [16]:
best = max(
    weekly_fov_analysis,
    key=lambda n: n['recommended_objects'][0]['recommendation_score'] if n['recommended_objects'] else 0,
)

print(f"Best night: {best['date']}\n")

print("=== FoV Fit Breakdown ===")
fit_counts = Counter(obj['fov_analysis']['fov_fit'] for obj in best['recommended_objects'])
for fit, count in fit_counts.most_common():
    print(f"  {fit:18}: {count}")

print("\n=== Top Recommendations with FoV Analysis ===")
for obj in best['recommended_objects'][:10]:
    fa = obj['fov_analysis']
    print(
        f"  {obj['name']:10} "
        f"score={obj['recommendation_score']:.3f}  "
        f"fov_fit={fa['fov_fit']:18} "
        f"{fa.get('note', '')}"
    )

Best night: 2026-07-24

=== FoV Fit Breakdown ===
  no_camera         : 15

=== Top Recommendations with FoV Analysis ===
  Jupiter    score=0.928  fov_fit=no_camera          No camera on file for this user — add camera specs in notebook 01 to enable FoV analysis.
  Mercury    score=0.927  fov_fit=no_camera          No camera on file for this user — add camera specs in notebook 01 to enable FoV analysis.
  Mars       score=0.925  fov_fit=no_camera          No camera on file for this user — add camera specs in notebook 01 to enable FoV analysis.
  Venus      score=0.912  fov_fit=no_camera          No camera on file for this user — add camera specs in notebook 01 to enable FoV analysis.
  Saturn     score=0.906  fov_fit=no_camera          No camera on file for this user — add camera specs in notebook 01 to enable FoV analysis.
  Moon       score=0.826  fov_fit=no_camera          No camera on file for this user — add camera specs in notebook 01 to enable FoV analysis.
  NGC6995    score=0